# 05 Reproduce Paper Tables

Regenerates `results_main.csv` (Table 3 of the paper) and `results_morph.csv`
(Table 4) from the released metric matrices.

**Inputs:** `metrics_main.csv`, `metrics_morph.csv`  
**Outputs:** `results_main.csv`, `results_morph.csv`

> Fully re-runnable from this repository. Requires only pandas, numpy, scipy
> and statsmodels — no NLP models and no access to the book texts.


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.multitest import multipletests

DATA = os.environ.get("KIDLIT_DATA", "../data/")

main  = pd.read_csv(DATA + "metrics_main.csv")
morph = pd.read_csv(DATA + "metrics_morph.csv")
print(main.shape, morph.shape)
main.origin_type.value_counts()

## Helpers

Cohen's *d* uses the pooled standard deviation, as in the paper.


In [ ]:
def cohens_d(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.std(ddof=1) ** 2 + b.std(ddof=1) ** 2) / 2)

def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"

## Main block — Table 3

35 metrics in 8 groups, Mann-Whitney U, Benjamini-Hochberg FDR at q = 0.05
applied across the whole block.

`personal_pron_ratio` is identically zero in both groups, so its test is
degenerate (p = 1.0). It is kept inside the correction family because the
published analysis did so; dropping it rescales every adjusted p-value by
34/35 and changes no conclusion.


In [ ]:
BLOCK = [
    ("Volume", "n_words", "Word count"),
    ("Volume", "n_sentences", "Sentence count"),
    ("Volume", "pages", "Pages"),
    ("Lexical diversity", "ttr", "TTR"),
    ("Lexical diversity", "sttr", "STTR"),
    ("Lexical diversity", "hapax_ratio", "Hapax ratio"),
    ("Lexical diversity", "yule_k", "Yule K index"),
    ("Lexical diversity", "shannon_h", "Shannon entropy"),
    ("Lexical diversity", "top1000_ratio", "Top-1000 ratio"),
    ("POS profile", "pos_noun_ratio", "Nouns"),
    ("POS profile", "pos_verb_ratio", "Verbs"),
    ("POS profile", "pos_adj_ratio", "Adjectives"),
    ("POS profile", "pos_adv_ratio", "Adverbs"),
    ("POS profile", "pos_propn_ratio", "Proper nouns"),
    ("POS profile", "pos_pron_ratio", "Pronouns"),
    ("POS profile", "personal_pron_ratio", "Personal pronouns"),
    ("POS profile", "animate_noun_ratio", "Animate nouns"),
    ("Syntax", "avg_tree_depth", "Mean dependency tree depth"),
    ("Syntax", "predicate_coord_ratio", "Predicate-coordination ratio"),
    ("Readability", "flesch_ru", "Flesch (RU)"),
    ("Readability", "fog_index", "FOG index"),
    ("Readability", "ari", "ARI"),
    ("Readability", "coleman_liau", "Coleman-Liau"),
    ("Word & sentence", "avg_word_len", "Mean word length (chars)"),
    ("Word & sentence", "avg_syllables", "Mean syllables per word"),
    ("Word & sentence", "avg_sent_words", "Mean sentence length (words)"),
    ("Word & sentence", "avg_sent_chars", "Mean sentence length (chars)"),
    ("Grammar", "past_tense_ratio", "Past tense"),
    ("Grammar", "present_tense_ratio", "Present tense"),
    ("Grammar", "future_tense_ratio", "Future tense"),
    ("Grammar", "imperative_ratio", "Imperative mood"),
    ("Grammar", "quest_sent_ratio", "Interrogative sentences"),
    ("Grammar", "excl_sent_ratio", "Exclamatory sentences"),
    ("Additional", "active_voice_ratio", "Active voice ratio"),
    ("Additional", "punct_diversity", "Punctuation diversity"),
]

o = main[main.origin_type == "RUS-O"]
t = main[main.origin_type == "RUS-T"]

rows = []
for grp, col, label in BLOCK:
    a, b = o[col], t[col]
    degenerate = a.nunique() == 1 and b.nunique() == 1 and a.iloc[0] == b.iloc[0]
    rows.append(dict(
        group=grp, metric_id=col, metric_label=label,
        n_rus_o=len(a), n_rus_t=len(b),
        mean_rus_o=a.mean(), mean_rus_t=b.mean(),
        sd_rus_o=a.std(ddof=1), sd_rus_t=b.std(ddof=1),
        cohens_d=0.0 if degenerate else cohens_d(a, b),
        p_ttest=1.0 if degenerate else ttest_ind(a, b).pvalue,
        p_mwu=1.0 if degenerate else mannwhitneyu(a, b).pvalue,
    ))

res = pd.DataFrame(rows)
res["p_adj_bh"]   = multipletests(res.p_mwu, method="fdr_bh")[1]
res["p_adj_holm"] = multipletests(res.p_mwu, method="holm")[1]
res["significance_bh"] = res.p_adj_bh.map(stars)
res["higher_in"] = np.where(res.cohens_d > 0, "RUS-O",
                            np.where(res.cohens_d < 0, "RUS-T", ""))

res[res.p_adj_bh < .05][["metric_label", "mean_rus_o", "mean_rus_t",
                         "cohens_d", "p_adj_bh"]]

Seven effects survive the correction: three of volume, three of lexical
diversity, one of syntax. This is Table 2 of the paper.


## Extended morphological block — Table 4

17 indices, two-sample *t*-test, Benjamini-Hochberg applied independently of
the main block.


In [ ]:
LABEL = {
    "analyticity_index": "Analyticity index (function words)",
    "verbality_index": "Verbality index",
    "substantivity_index": "Substantivity index",
    "pronominality_index": "Pronominality index",
    "adjectivity_index": "Adjectivity index",
    "nominal_vocab_index": "Nominal vocabulary index",
    "noun_verb_ratio": "Noun-to-verb ratio",
    "genitive_ratio": "Genitive case forms",
    "instrumental_ratio": "Instrumental case forms",
    "short_adj_ratio": "Short adjectives",
    "full_participle_ratio": "Full participles",
    "short_participle_ratio": "Short participles",
    "predicative_ratio": "Predicatives",
    "gerund_ratio": "Gerunds",
    "infinitive_ratio": "Infinitives",
    "numeral_ratio": "Numerals",
    "particle_ratio": "Particles",
}

mo = morph[morph.origin_type == "RUS-O"]
mt = morph[morph.origin_type == "RUS-T"]

rows = []
for col, label in LABEL.items():
    a, b = mo[col], mt[col]
    tt = ttest_ind(a, b)
    rows.append(dict(
        metric_id=col, metric_label=label, n_rus_o=len(a), n_rus_t=len(b),
        mean_rus_o=a.mean(), mean_rus_t=b.mean(),
        sd_rus_o=a.std(ddof=1), sd_rus_t=b.std(ddof=1),
        t_statistic=tt.statistic, cohens_d=cohens_d(a, b),
        p_ttest=tt.pvalue, p_mwu=mannwhitneyu(a, b).pvalue,
    ))

rm = pd.DataFrame(rows)
rm["p_adj_bh"]   = multipletests(rm.p_ttest, method="fdr_bh")[1]
rm["p_adj_holm"] = multipletests(rm.p_ttest, method="holm")[1]
rm["significance_bh"] = rm.p_adj_bh.map(stars)
rm["higher_in"] = np.where(rm.cohens_d > 0, "RUS-O", "RUS-T")
rm = rm.sort_values("p_adj_bh").reset_index(drop=True)

print("smallest adjusted p:", round(rm.p_adj_bh.min(), 4))
rm[["metric_label", "mean_rus_o", "mean_rus_t", "cohens_d", "p_adj_bh"]].head()

No morphological index survives the correction. This is the negative result
of Section 5.4.


## Write the results tables


In [ ]:
for c in ["mean_rus_o", "mean_rus_t", "sd_rus_o", "sd_rus_t"]:
    res[c] = res[c].round(4); rm[c] = rm[c].round(4)
res["cohens_d"] = res.cohens_d.round(3)
rm[["t_statistic", "cohens_d"]] = rm[["t_statistic", "cohens_d"]].round(3)
for c in ["p_ttest", "p_mwu", "p_adj_bh", "p_adj_holm"]:
    res[c] = res[c].round(6); rm[c] = rm[c].round(6)
res.loc[res.metric_id == "personal_pron_ratio", "higher_in"] = ""

res.to_csv(DATA + "results_main.csv", index=False, encoding="utf-8", lineterminator="\n")
rm.to_csv(DATA + "results_morph.csv", index=False, encoding="utf-8", lineterminator="\n")
print("written:", res.shape, rm.shape)